#  SuperAI Hackathon 2026: FahMai RAG Pipeline
**Clean and Optimized RAG Pipeline using Hybrid Search, Entity Boosting, and the KBTG Model.**


---
**Pipeline Steps:**

1. **No Chunking:** Load full documents to preserve table structures perfectly.
2. **Hybrid Search:** Combine Keyword (BM25) and Semantic (E5) search using RRF.
3. **Target Boosting:** Extract product names from questions and apply a **10x score** to exact-match documents.
4. **Smart Filter:** Block off-topic questions early to save execution time.
5. **LLM (KBTG):** Feed the Top 3 docs to the KBTG model, equipped with auto-retry for server errors.

---
## Cell 0 — Setup: Unzip Data

In [ ]:
!unzip data.zip

Archive:  data.zip
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT-005_daonuea_all_in_one_24.md  
  inflating: data/knowledge_base/products/DN-LT-001_daonuea_airbook_15.md  
  inflating: data/knowledge_bas

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 55.1 MB/s eta 0:00:00


---
## Cell 2 — Configuration



In [ ]:
import os, csv, re, time, requests ,json
import numpy as np
from pathlib import Path
from collections import Counter
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import torch
from google.colab import userdata

N_QUESTIONS = 100

TOP_K = 3
RRF_K = 60
DATA_DIR = "/content/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

PRIMARY_MODEL = "kbtg"
ENSEMBLE_MODELS = ["kbtg", "openthaigpt", "pathumma"]
EMBED_MODEL_NAME = "intfloat/multilingual-e5-large"
USE_ENSEMBLE = False

THAILLM_API_KEY = userdata.get('ThaiLLM')
print(f" API Key loaded: {'Yes' if THAILLM_API_KEY else 'No'}")
print(f" Model: {PRIMARY_MODEL} | TOP_K: {TOP_K}")

 API Key loaded: Yes
 Model: kbtg | TOP_K: 3


---
## Cell 3 — LLM Client

`ask_llm()` → เรียก ThaiLLM API พร้อม retry + rate limit handling  
`parse_answer()` → ดึงตัวเลขคำตอบ 1-10 จาก response (รองรับ `<think>` tags)

In [ ]:
def ask_llm(messages, model="typhoon", max_retries=7, temperature=0):
    """Call ThaiLLM API with retry and rate-limit handling."""
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2048,
        "temperature": temperature,
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=180)
            if resp.status_code == 429:
                wait = min(2 ** attempt, 60)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            if resp.status_code >= 500:
                wait = min(2 ** attempt, 30)
                print(f"  Server error {resp.status_code}, retrying in {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)
    return None


def parse_answer(text):
    """Extract answer number (1-10) from LLM response."""
    if text is None:
        return None
    # Remove <think>...</think> blocks (Pathumma/OpenThaiGPT)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # Pattern 1: ANSWER: X (most reliable)
    m = re.search(r"ANSWER\s*[:：]\s*(\d{1,2})\b", clean, re.IGNORECASE)
    if m:
        val = int(m.group(1))
        if 1 <= val <= 10: return val

    # Pattern 2: Thai patterns — คำตอบคือ X, ตอบข้อ X
    m = re.search(r"(?:คำตอบ|ตอบ|เลือก)\s*(?:คือ|:)?\s*(?:ข้อ\s*)?(\d{1,2})\b", clean)
    if m:
        val = int(m.group(1))
        if 1 <= val <= 10: return val

    # Pattern 3: Last standalone number 1-10
    nums = re.findall(r"\b(\d{1,2})\b", clean)
    for n in reversed(nums):
        if 1 <= int(n) <= 10: return int(n)

    return None

print(" ask_llm() and parse_answer() defined")

 ask_llm() and parse_answer() defined


---
## Cell 4 — Load Questions

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f" Loaded {len(questions)} questions")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in list(questions[0]['choices'].items())[:3]:
    print(f"  {k}. {v[:80]}")

 Loaded 100 questions

Example — Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  1. 3 ATM
  2. IP68
  3. 5 ATM


---
## Cell 5 — Load Knowledge Base (Whole Documents)

**ไม่ chunk!** Product docs เฉลี่ย 2,953 chars — เล็กพอที่จะใช้ทั้ง doc  
ป้องกันปัญหา spec table หลุดจาก product name

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    rel_path = str(fp.relative_to(kb_dir))

    if 'products/' in rel_path:     category = 'product'
    elif 'policies/' in rel_path:   category = 'policy'
    elif 'store_info/' in rel_path: category = 'store_info'
    else:                           category = 'other'

    title_match = re.match(r'^#\s*(.+)', text)
    title = title_match.group(1).strip() if title_match else fp.stem

    code_match = re.match(r'^([A-Z]{2}-[A-Z]{2,3}-\d{3})', fp.stem)
    product_code = code_match.group(1) if code_match else ""

    price_match = re.search(r'ราคา:\s*฿?([\d,]+)', text)
    price = int(price_match.group(1).replace(',', '')) if price_match else 0

    documents.append({
        "path": rel_path, "text": text, "category": category,
        "title": title, "code": product_code, "price": price,
    })

print(f" Loaded {len(documents)} documents (NO chunking — whole docs)")
for cat in ['product', 'policy', 'store_info']:
    n = sum(1 for d in documents if d['category']==cat)
    sizes = [len(d['text']) for d in documents if d['category']==cat]
    print(f"  {cat}: {n} docs, avg {sum(sizes)//n} chars")

 Loaded 118 documents (NO chunking — whole docs)
  product: 110 docs, avg 2953 chars
  policy: 5 docs, avg 6576 chars
  store_info: 3 docs, avg 8305 chars


---
## Cell 6 — Product Name Index

สร้าง mapping ชื่อสินค้า → doc index หลายรูปแบบ:
- ชื่อเต็ม Thai + English
- ชื่อย่อ (เช่น "X9 Pro", "AirBook 14")
- Product code (เช่น "SF-SP-002")
- ชื่อจาก filename

In [ ]:
def build_product_index(documents):
    """Build extensive product name → doc index mapping."""
    name_to_idx = {}
    for idx, doc in enumerate(documents):
        if doc['category'] != 'product': continue
        title = doc['title']
        path = doc['path']

        # 1. Full title
        name_to_idx[title.lower()] = idx

        # 2. English name from parentheses
        eng_match = re.search(r'\(([^)]+)\)', title)
        if eng_match:
            eng_name = eng_match.group(1).strip()
            name_to_idx[eng_name.lower()] = idx
            parts = eng_name.split()
            if len(parts) > 1:
                name_to_idx[' '.join(parts[1:]).lower()] = idx

        # 3. Thai name only
        thai_name = re.sub(r'\s*\(.*\)', '', title).strip()
        name_to_idx[thai_name.lower()] = idx
        thai_parts = thai_name.split()
        if len(thai_parts) > 1:
            name_to_idx[' '.join(thai_parts[1:]).lower()] = idx

        # 4. From filename
        fname = path.split('/')[-1].replace('.md', '')
        parts = fname.split('_', 1)
        if len(parts) > 1:
            name_to_idx[parts[1].replace('_', ' ').lower()] = idx

        # 5. Product code
        if doc['code']:
            name_to_idx[doc['code'].lower()] = idx

        # 6. Model identifiers (e.g. "X9 Pro", "S3 Ultra")
        model_pats = re.findall(r'[A-Za-z]+\s*(?:\d+[A-Za-z]*\s*(?:Pro|Max|Ultra|SE|Lite|Plus|Mini)?)', title, re.IGNORECASE)
        for mp in model_pats:
            name_to_idx[mp.strip().lower()] = idx

    return name_to_idx


product_index = build_product_index(documents)
print(f" Product index: {len(product_index)} name entries")
# Show samples
for name in list(product_index.keys())[:5]:
    print(f"  '{name}' → doc[{product_index[name]}] {documents[product_index[name]]['title'][:40]}")

 Product index: 709 name entries
  'อาร์คเวฟ proview 27 4k (arcwave proview 27 4k)' → doc[5] อาร์คเวฟ ProView 27 4K (ArcWave ProView 
  'arcwave proview 27 4k' → doc[5] อาร์คเวฟ ProView 27 4K (ArcWave ProView 
  'proview 27 4k' → doc[5] อาร์คเวฟ ProView 27 4K (ArcWave ProView 
  'อาร์คเวฟ proview 27 4k' → doc[5] อาร์คเวฟ ProView 27 4K (ArcWave ProView 
  'aw-mn-001' → doc[5] อาร์คเวฟ ProView 27 4K (ArcWave ProView 


---
## Cell 7 — Dense Embeddings (Whole Docs)

ใช้ `intfloat/multilingual-e5-large` — encode ทั้ง 118 docs  
ใช้ prefix `"passage: "` สำหรับ docs, `"query: "` สำหรับ questions

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(EMBED_MODEL_NAME)
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

doc_texts_for_embed = []
for doc in documents:
    meta = f"[{doc['category']}] [{doc['title']}] "
    doc_texts_for_embed.append(PASSAGE_PREFIX + meta + doc['text'])

print(f"Embedding {len(doc_texts_for_embed)} whole documents...")
doc_embeddings = embed_model.encode(
    doc_texts_for_embed, batch_size=16,
    show_progress_bar=True, normalize_embeddings=True
)
print(f"✅ Doc embeddings shape: {doc_embeddings.shape}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 118 whole documents...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Doc embeddings shape: (118, 1024)


---
## Cell 8 — BM25 Index

ใช้ `pythainlp` tokenize ภาษาไทย → BM25 จับ keyword matching  
สำคัญมากสำหรับชื่อสินค้าเช่น "SaiFah", "คลื่นเสียง", "วงโคจร"

In [ ]:
def thai_tokenize(text):
    """Tokenize Thai+English text using pythainlp newmm."""
    tokens = word_tokenize(text, engine="newmm")
    return [t for t in tokens if t.strip()]

print("Tokenizing documents for BM25...")
tokenized_docs = [thai_tokenize(d['text']) for d in documents]
bm25 = BM25Okapi(tokenized_docs)
print(f"BM25 index built over {len(tokenized_docs)} documents")

Tokenizing documents for BM25...
BM25 index built over 118 documents


---
## Cell 9 — Smart Retrieval

หัวใจของ pipeline — retrieval ที่ฉลาดขึ้น:

| Feature | ทำอะไร |
|---------|--------|
| `dense_retrieve()` | Semantic search ด้วย E5-Large |
| `bm25_retrieve()` | Keyword search ด้วย BM25 |
| `extract_mentioned_products()` | ดึงชื่อสินค้าจากข้อความ |
| `is_comparison_question()` | ตรวจว่าเป็นคำถามเปรียบเทียบไหม |
| `is_calculation_question()` | เอาไว้ดักคำที่เกี่ยวกับการคำนวณราคาหรือพอยท์ |
| `smart_retrieve()` | **รวมทุกอย่าง** — entity matching + multi-doc + policy routing + RRF |

In [ ]:
def dense_retrieve(query, k=TOP_K):
    """Semantic search using E5-Large embeddings."""
    q_emb = embed_model.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    scores = np.dot(doc_embeddings, q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]


def bm25_retrieve(query, k=TOP_K):
    """Keyword search using BM25 with Thai tokenization."""
    tokens = thai_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]


def extract_mentioned_products(text, product_index):
    """Find all product names mentioned in text."""
    text_lower = text.lower()
    matches = []
    sorted_names = sorted(product_index.keys(), key=len, reverse=True)
    matched_indices = set()
    for name in sorted_names:
        if len(name) < 4: continue
        if name in text_lower:
            doc_idx = product_index[name]
            if doc_idx not in matched_indices:
                matches.append((name, doc_idx))
                matched_indices.add(doc_idx)
    return matches


def is_comparison_question(question):
    """Detect comparison between 2+ products."""
    signals = ['กับ', 'เปรียบเทียบ', 'ต่างกัน', 'เหมือนกัน',
               'ตัวไหน', 'อันไหน', 'ดีกว่า', 'ไหนดี',
               'vs', 'เทียบ', 'ทั้งสอง', 'ทั้งคู่']
    return any(s in question for s in signals)


def is_calculation_question(question):
    """Detect price calculation questions."""
    signals = ['ราคารวม', 'รวมเท่าไหร่', 'ทั้งหมดเท่าไหร่',
               'กี่ FahMai Points', 'กี่ points', 'กี่พอยท์',
               'ใช้ Points ลด', 'งบ', 'มีเงิน']
    return any(s in question.lower() for s in signals)


def smart_retrieve(question, choices, k=TOP_K):
    """
    Enhanced retrieval pipeline:
    1. Extract product names from question AND choices → force-include docs
    2. For comparison Qs → ensure both products' docs are included
    3. Policy keyword routing → force-include relevant policy docs
    4. Store/FAQ routing → include store_info docs
    5. Hybrid BM25+Dense with RRF for remaining slots
    """
    # === Step 1: Entity extraction from question ===
    mentioned = extract_mentioned_products(question, product_index)
    forced_indices = set(idx for _, idx in mentioned)

    # === Step 2: Entity extraction from choices 1-8 ===
    for cid in range(1, 9):
        ctext = str(choices.get(str(cid), ''))
        if len(ctext) > 3:
            choice_mentions = extract_mentioned_products(ctext, product_index)
            for _, idx in choice_mentions[:1]:
                forced_indices.add(idx)

    # === Step 3: Comparison → get 2+ product docs ===
    if is_comparison_question(question) and len(forced_indices) < 2:
        b_idx, b_scores = bm25_retrieve(question, k=10)
        for bi in b_idx:
            if documents[bi]['category'] == 'product' and bi not in forced_indices:
                forced_indices.add(bi)
                if len(forced_indices) >= 3: break

    # === Step 4: Policy keyword routing ===
    policy_kw = {
        'คืน': 'return_policy', 'คืนสินค้า': 'return_policy',
        'ประกัน': 'warranty_policy', 'เคลม': 'warranty_policy', 'Care+': 'warranty_policy',
        'จัดส่ง': 'shipping_policy', 'ส่ง': 'shipping_policy',
        'ยกเลิก': 'cancellation_policy', 'pre-order': 'cancellation_policy',
        'สมาชิก': 'membership_points_policy', 'Points': 'membership_points_policy',
        'พอยท์': 'membership_points_policy', 'point': 'membership_points_policy',
    }
    q_lower = question.lower()
    for kw, policy_file in policy_kw.items():
        if kw.lower() in q_lower:
            for i, d in enumerate(documents):
                if policy_file in d['path']:
                    forced_indices.add(i)

    # === Step 5: Store/FAQ routing ===
    store_kw = ['ฟ้าใหม่', 'ร้าน', 'สาขา', 'ชำระ', 'จ่าย', 'สมัคร']
    if any(kw in question for kw in store_kw):
        for i, d in enumerate(documents):
            if d['category'] == 'store_info':
                forced_indices.add(i)

    # === Step 6: Hybrid RRF for remaining slots ===
    remaining_k = max(k - len(forced_indices), 3)
    d_idx, _ = dense_retrieve(question, k=remaining_k * 3)
    b_idx, _ = bm25_retrieve(question, k=remaining_k * 3)

    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (RRF_K + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (RRF_K + rank)
    for idx in forced_indices:
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 0.1  # Heavy boost

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    result_indices = list(forced_indices)
    for idx, score in sorted_docs:
        if idx not in result_indices:
            result_indices.append(idx)
        if len(result_indices) >= k: break

    return [documents[i] for i in result_indices[:k]]

print(" Smart retrieval functions defined")
print(f" dense_retrieve(), bm25_retrieve(), extract_mentioned_products()")
print(f" is_comparison_question(), is_calculation_question(), smart_retrieve()")

 Smart retrieval functions defined
 dense_retrieve(), bm25_retrieve(), extract_mentioned_products()
 is_comparison_question(), is_calculation_question(), smart_retrieve()


---
## Cell 10 — Off-Topic Router (FIXED)


In [ ]:
def is_off_topic(question):

    q_lower = question.lower()

    # Check 1: มีชื่อสินค้าในคำถามไหม?
    mentioned = extract_mentioned_products(question, product_index)
    if mentioned: return False

    # Check 2: FahMai-related signals
    fahmai_signals = [
        'ฟ้าใหม่', 'fahmai', 'สายฟ้า', 'saifah', 'ดาวเหนือ', 'daonuea',
        'คลื่นเสียง', 'kluensiang', 'วงโคจร', 'wongkhojon', 'จุดเชื่อม', 'judchuam',
        'novatech', 'pulsegear', 'arcwave', 'zenbyte',
        'สินค้า', 'ร้าน', 'สาขา', 'ประกัน', 'care+',
        'คืนสินค้า', 'จัดส่ง', 'สมาชิก', 'points', 'พอยท์',
        'โทรศัพท์', 'มือถือ', 'แล็ปท็อป', 'โน้ตบุ๊ค',
        'หูฟัง', 'นาฬิกา', 'แท็บเล็ต', 'ลำโพง', 'จอ',
        'watch', 'phone', 'book', 'buds', 'tab', 'dock',
        'hub', 'charger', 'case', 'pen', 'keyboard', 'mouse',
        'pro', 'max', 'ultra', 'soundbar', 'headpro', 'headon',
        'ชาร์จ', 'แบต', 'กันน้ำ', 'atm', 'ram', 'ssd',
        'ราคา', 'รุ่น', 'bluetooth', 'wifi', 'gps', 'nfc',
        'amoled', 'oled', 'lcd', 'กล้อง', 'เซ็นเซอร์',
        'เคส', 'ฟิล์ม', 'สาย', 'แท่น', 'อะแดปเตอร์',
        'gold', 'platinum', 'คืน', 'เคลม', 'ซ่อม', 'ผ่อน',
        'pre-order', 'on-site', 'mega sale',
        'สมาร์ท', 'ไร้สาย', 'wireless', 'anc', 'ldac',
        'ซื้อ', 'สั่งซื้อ', 'สั่ง', 'order',
        'stormbook', 'airbook', 'probook', 'studybook', 'creatorbook', 'flexbook',
        'tower', 'mini pc', 'all-in-one',
        'band', 'ring', 'sport', 'classic',
        'soundpillar', 'boombox', 'homepod',
        'qipad', 'powerbank', 'power bank',
        'slimbook', 'novabuds', 'proview',
    ]
    if any(s in q_lower for s in fahmai_signals): return False

    # Check 3: Only if NO signals → check off-topic patterns
    off_topic_clear = [
        'สูตรอาหาร', 'สูตรทำ', 'วิธีทำอาหาร', 'ผัดกระเพรา',
        'ตั๋วเครื่องบิน', 'วันหยุดราชการ', 'วันหยุด',
        'ดอกเบี้ยเงินฝาก', 'อัตราแลกเปลี่ยน',
        'นายกรัฐมนตรี', 'การเลือกตั้ง',
        'พยากรณ์อากาศ', 'หวย', 'ลอตเตอรี่',
        'ฟุตบอลโลก', 'โอลิมปิก',
    ]
    if any(kw in q_lower for kw in off_topic_clear): return True

    return False  # Default: NOT off-topic

print(" is_off_topic() defined (FIXED — won't misclassify Q82)")

 is_off_topic() defined (FIXED — won't misclassify Q82)


---
## Cell 11 — Prompt Engineering (CoT + Few-shot)

- **System prompt:** วิเคราะห์ทีละขั้น ให้ความสำคัญ 1-8 ก่อน 9/10
- **Few-shot examples:** สอน model ว่า format คำตอบเป็นยังไง
- **`build_rag_prompt()`:** ประกอบ context + question + choices

In [ ]:
SYSTEM_PROMPT = """คุณเป็นผู้ช่วยตอบคำถามของร้าน FahMai (ฟ้าใหม่) ร้านขายอุปกรณ์อิเล็กทรอนิกส์ในประเทศไทย
คุณจะได้รับข้อมูลอ้างอิงจากฐานความรู้ของร้าน และต้องเลือกคำตอบจากตัวเลือกที่ให้มา

วิธีตอบ:
1. อ่านข้อมูลอ้างอิงทั้งหมดอย่างละเอียด
2. วิเคราะห์ว่าคำถามถามเกี่ยวกับอะไร
3. หาหลักฐาน/ข้อมูลที่ตรงกับคำถามในเอกสาร
4. เปรียบเทียบแต่ละตัวเลือก (1-8) กับข้อมูลที่พบ
5. เลือกตัวเลือกที่ตรงกับข้อมูลมากที่สุด

กฎสำคัญ:
- ตัวเลือก 1-8 คือคำตอบที่เป็นเนื้อหา ให้ความสำคัญกับตัวเลือกเหล่านี้ก่อน
- ตัวเลือก 9 "ไม่มีข้อมูล" ใช้เมื่อคำถามเกี่ยวกับ FahMai จริงๆ แต่ค้นทุกเอกสารแล้วไม่พบคำตอบ
- ตัวเลือก 10 "ไม่เกี่ยวข้อง" ใช้เฉพาะเมื่อคำถามไม่เกี่ยวกับสินค้า/นโยบาย/ร้าน FahMai อย่างสิ้นเชิง
- ถ้าคำถามเป็นการเปรียบเทียบ ให้ดูข้อมูลจากทั้งสองรุ่น
- ถ้าคำถามเกี่ยวกับราคา/ตัวเลข ให้ตรวจสอบตัวเลขให้ตรงกับเอกสาร
- ถ้าคำถามต้องคำนวณ ให้คำนวณทีละขั้นตอน

ตอบเพียงบรรทัดเดียว: ANSWER: X (X = 1-10)"""


FEW_SHOT = """ตัวอย่าง:
Q: Watch S3 Ultra กันน้ำได้กี่ ATM?
เอกสาร: "กันน้ำ | 100 เมตร (10 ATM)"
ตัวเลือก: 1) 3 ATM 2) IP68 3) 5 ATM 4) IP67 5) 10 ATM ...
วิเคราะห์: เอกสารระบุ 10 ATM → ตรงกับข้อ 5
ANSWER: 5

Q: กรุงเทพมีกี่เขต?
วิเคราะห์: คำถามไม่เกี่ยวกับ FahMai → ข้อ 10
ANSWER: 10

Q: SaiFah Phone Z99 น้ำหนักเท่าไหร่?
วิเคราะห์: ไม่มีรุ่น Z99 ในเอกสาร แต่คำถามเกี่ยวกับสินค้า FahMai → ข้อ 9
ANSWER: 9
"""


def build_rag_prompt(question, choices, retrieved_docs):
    """Build RAG prompt with context + CoT + few-shot."""
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        text = doc['text']
        if len(text) > 3500:
            text = text[:3500] + "\n... (ตัดทอน)"
        context_parts.append(f"=== เอกสารที่ {i}: {doc['title']} ({doc['category']}) ===\n{text}")

    context = "\n\n".join(context_parts)
    choices_text = "\n".join(f"{k}) {v}" for k, v in sorted(choices.items(), key=lambda x: int(x[0])))

    return f"""{FEW_SHOT}

--- ข้อมูลอ้างอิง ---
{context}

--- คำถาม ---
{question}

--- ตัวเลือก ---
{choices_text}

วิเคราะห์สั้นๆ แล้วตอบ ANSWER: X"""

print("SYSTEM_PROMPT, FEW_SHOT, build_rag_prompt() defined")

SYSTEM_PROMPT, FEW_SHOT, build_rag_prompt() defined


---
## Cell 12 — Pipeline (Main)

`run_pipeline_v3()` — ทำทุกอย่างต่อ 1 model:
1. Off-topic check
2. Smart retrieval
3. Build prompt + call LLM
4. Parse answer

In [ ]:
def run_pipeline_v3(questions, model=PRIMARY_MODEL, n=N_QUESTIONS):

    predictions = {}
    failed = []

    with open("submission_backup.csv", "w", encoding="utf-8") as f:
        f.write("id,answer\n")

        for i, q in enumerate(questions[:n]):
            qid = q["id"]
            question = q["question"]
            choices = q["choices"]

            # --- Step 1: Off-topic check ---
            if is_off_topic(question):
                pred = 10
                predictions[qid] = pred
                f.write(f"{qid},{pred}\n")
                f.flush() # บังคับเซฟลงดิสก์ทันที
                print(f"  [{i+1:>3}/{n}] Q{qid:>3}: pred=10  (ROUTED: off-topic)")
                continue

            # --- Step 2: Smart Retrieval ---
            retrieved = smart_retrieve(question, choices, TOP_K)

            # --- Step 3: Build prompt & generate ---
            prompt = build_rag_prompt(question, choices, retrieved)
            raw = ask_llm([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ], model=model, max_retries=10) # เพิ่ม Retries

            # --- Step 4: Parse answer ---
            pred = parse_answer(raw)
            if pred is None:
                failed.append(qid)
                pred = 1 # เดา 1 ไว้ก่อนดีกว่าส่งค่าว่าง

            predictions[qid] = pred
            f.write(f"{qid},{pred}\n")
            f.flush() # เซฟทันที

            sources = [d['title'][:25] for d in retrieved[:3]]
            print(f"  [{i+1:>3}/{n}] Q{qid:>3}: pred={pred}  src={sources}")

            time.sleep(3)

    if failed:
        print(f"\n⚠ Failed to parse {len(failed)} answers: {failed}")
    return predictions

print("run_pipeline_v3() defined")

run_pipeline_v3() defined


---
## Cell 13 — Ensemble Voting (3 Models)

Majority vote — ถ้า 2/3 เห็นตรงกัน ใช้เสียงข้างมาก  
**Tiebreak (3-way split):** ใช้ **KBTG**

In [ ]:
def run_ensemble_v3(questions, models=ENSEMBLE_MODELS, n=N_QUESTIONS):
    """Run multiple models and majority-vote. Tiebreak: OpenThaiGPT."""
    all_preds = {}
    for model in models:
        print(f"\n{'='*60}")
        print(f"Running model: {model}")
        print(f"{'='*60}")
        preds = run_pipeline_v3(questions, model=model, n=n)
        all_preds[model] = preds

    # Majority voting
    ensemble_preds = {}
    disagreements = 0

    for q in questions[:n]:
        qid = q["id"]
        votes = [all_preds[m].get(qid, 1) for m in models]
        vote_counts = Counter(votes)
        best_answer = vote_counts.most_common(1)[0][0]

        # ★ TIEBREAK: 3-way split
        if vote_counts.most_common(1)[0][1] == 1:
            best_answer = all_preds["kbtg"].get(qid, 1)

        if len(set(votes)) > 1:
            disagreements += 1
        ensemble_preds[qid] = best_answer

    print(f"\nEnsemble: {len(models)} models, {disagreements} disagreements / {n}")

    # Show disagreements
    if disagreements > 0:
        print("\nDisagreements:")
        for q in questions[:n]:
            qid = q["id"]
            votes = {m: all_preds[m].get(qid) for m in models}
            if len(set(votes.values())) > 1:
                print(f"  Q{qid}: {votes} → {ensemble_preds[qid]}")

    return ensemble_preds, all_preds

print("run_ensemble_v3() defined (tiebreak = kbtg)")

run_ensemble_v3() defined (tiebreak = kbtg)


---
## Cell 14 — ▶️ RUN PIPELINE

เลือก `USE_ENSEMBLE = True` เพื่อ run 3 models + vote  
หรือ `USE_ENSEMBLE = False` เพื่อ run model ตัวเดียว

In [ ]:
if USE_ENSEMBLE:
    print("🚀 Running ENSEMBLE mode (3 models)...\n")
    best_preds, all_model_preds = run_ensemble_v3(questions)
else:
    print(f"🚀 Running SINGLE model: {PRIMARY_MODEL}...\n")
    best_preds = run_pipeline_v3(questions, model=PRIMARY_MODEL)

🚀 Running SINGLE model: kbtg...

  [  1/100] Q  1: pred=5  src=['วงโคจร Watch S3 Ultra (Wo', 'วงโคจร Watch S3 (WongKhoJ', 'วงโคจร Watch S3 Pro (Wong']
  [  2/100] Q  2: pred=7  src=['จุดเชื่อม ปากกา SaiFah Pe', 'สายฟ้า แท็บ Draw Pro (Sai', 'สายฟ้า แท็บ Mini 7 (SaiFa']
  [  3/100] Q  3: pred=2  src=['คลื่นเสียง เฮดโปร X1 (Klu', 'คลื่นเสียง เฮดโปร X1 SE (', 'คลื่นเสียง บัดส์ สปอร์ต ไ']
  [  4/100] Q  4: pred=6  src=['เกี่ยวกับฟ้าใหม่ (About F', 'คู่มือเลือกซื้อสมาร์ทโฟน ', 'คำถามที่พบบ่อย (General F']
  [  5/100] Q  5: pred=6  src=['เกี่ยวกับฟ้าใหม่ (About F', 'คู่มือเลือกซื้อสมาร์ทโฟน ', 'คำถามที่พบบ่อย (General F']
  [  6/100] Q  6: pred=8  src=['เกี่ยวกับฟ้าใหม่ (About F', 'คู่มือเลือกซื้อสมาร์ทโฟน ', 'คำถามที่พบบ่อย (General F']
  Server error 520, retrying in 1s...
  [  7/100] Q  7: pred=1  src=['ดาวเหนือ แอร์บุ๊ก 14 รุ่น', 'ดาวเหนือ แอร์บุ๊ก 14 (Dao', 'ดาวเหนือ สตั๊ดดี้บุ๊ก 14 ']
  [  8/100] Q  8: pred=4  src=['สายฟ้า โฟน DuoPad (SaiFah', 'คำถามที่พบบ่อย (General F', 'สายฟ้า โฟน Po

---
## Cell 15 — Save Submission CSV

In [ ]:
output_file = "submission.csv"
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        answer = best_preds.get(qid, 1)
        writer.writerow([qid, answer])

print(f"✅ {output_file} written ({len(questions)} rows)")
dist = Counter(best_preds.values())
print(f"\nAnswer distribution:")
for k in sorted(dist.keys()):
    print(f"  Choice {k}: {dist[k]}")

✅ submission.csv written (100 rows)

Answer distribution:
  Choice 1: 8
  Choice 2: 10
  Choice 3: 11
  Choice 4: 15
  Choice 5: 12
  Choice 6: 12
  Choice 7: 10
  Choice 8: 7
  Choice 9: 12
  Choice 10: 3
